# Lab Assignment 2
## Trigram Language Models: Add-k Smoothing and Linear Interpolation

---

### Objective

Building on the Bigram model from the previous lab, you will implement a **Trigram (N=3)**
language model, evaluate it with **perplexity**, and then improve it.

The tutorial showed you Add-1 smoothing. Here you go two steps further:

1. you implement the general **Add-k** smoothing and tune `k`,
2. you then build a **linear interpolation** model, which combines the trigram, bigram and
   unigram estimates together, and tune its weights.

Finally you compare the two models and report which one is better.

### Tasks

| Task | What you implement |
|---|---|
| **Task 1** | Preprocessing and counting for trigrams |
| **Task 2** | Trigram probability with **Add-k** smoothing |
| **Task 3** | Sentence log probability |
| **Task 4** | Perplexity |
| **Task 5** | **Linear interpolation** of trigram, bigram and unigram models |
| **Task 6** | Tune both models and compare them |

Run all cells top to bottom before submitting.

---
## Setup

Run this cell first. It gives you the training corpus and the test sentences.

In [ ]:
import math
import re
from collections import Counter

corpus_text = """
I want to eat chinese food for lunch.
I would like to order the house special.
The chinese food at this restaurant is very good.
I want to know what is on the menu for today.
Please bring me the bill when you are ready.
The food here is always fresh and delicious.
I am looking for a restaurant that serves authentic italian food.
I want to book a table for two people tonight.
What is the special dish for today?
The service we received was excellent and the waiter was friendly.
My friend wants to eat noodles for dinner.
The noodles here are the best in the city.
I would like some sparkling water please.
This is the best food I have ever had in my life.
I am very hungry and I want to eat now.
I want some dessert after the main course.
The dessert menu looks great, I would like to order one.
My friend is a vegetarian, so we need vegetarian options.
Do you have any good vegetarian options on the menu?
I would like to pay with a credit card if that is okay.
The restaurant is very clean and the atmosphere is lovely.
We want to sit by the window for a better view.
This italian dish is amazing, I will order it again.
The bill is on me tonight, please bring it to me.
I am so full, I cannot eat another bite.
We would like to start with an appetizer.
The soup of the day is tomato soup.
I will have the tomato soup.
My friend will have the chicken salad.
The chicken salad is a popular choice here.
Could we get some more bread please?
The bread is baked fresh in the restaurant.
I want a glass of red wine with my meal.
The red wine from this region is famous.
I am not very hungry, so I will just have a small salad.
We want to celebrate a special occasion.
This place is perfect for a celebration.
The chef here is from Italy.
That explains why the italian food is so authentic.
I want to try something new today.
What would you recommend from the menu?
The fish is the freshest catch of the day.
Then I will have the fish.
My friend wants the steak.
How would he like the steak cooked?
He wants the steak medium rare.
I want to order a pizza to go.
The pizza here is better than the pasta.
I am allergic to nuts, please be careful.
We will make sure there are no nuts in your food.
"""

corpus_sentences = [s.strip() for s in corpus_text.strip().split("\n") if s.strip()]

# Test sentences. These are NOT in the training corpus, and each contains at least
# one trigram the model has never seen.
test_sentences = [
    "I want to eat italian food",
    "My friend would like to eat pizza",
    "The waiter will bring the menu",
]

print(f"{len(corpus_sentences)} training sentences, {len(test_sentences)} test sentences")

---
# Task 1: Preprocessing and Counting

### 1a. Preprocessing

Adapt your preprocessing for a trigram model. For each sentence you must prepend **two**
start-of-sentence tokens (`<.s> <.s>`) and append **one** end-of-sentence token (`<./s>`).

Example: `"I am here"` becomes `['<.s>', '<.s>', 'i', 'am', 'here', '<./s>']`

**Justification:** this ensures every trigram has a valid two-word history. The probability of
the first word `'i'` is then `P(i | <.s>, <.s>)`.

**Hints**
- Lowercase, remove punctuation, then split on whitespace.
- Removing punctuation matters: otherwise `"food."` and `"food"` are two different words.
  `re.sub(r"[^a-z\s]", "", sentence.lower())` is one way.

### 1b. Counting

Generate and store counts for **unigrams (n=1)**, **bigrams (n=2)** and **trigrams (n=3)**.

You will need all three later: Task 2 uses the bigram counts as its denominator, and Task 5
uses all three at once.

**Hints**
- `extract_ngrams` should slide a window of width `n` and return **tuples** (tuples can be
  dictionary keys, lists cannot). The loop range is `len(tokens) - n + 1`.
- `counter.update(list_of_ngrams)` adds a whole list at once.
- `V` is the number of **distinct** tokens: `len(unigram_counts)`.
- `total_tokens` is the **total** number of tokens, i.e. `sum(unigram_counts.values())`.
  You will need it for the unigram probability in Task 5.

In [ ]:
START = "<.s>"
END = "<./s>"

def preprocess(sentence):
    """
    Tokenize and pad a sentence for a TRIGRAM model.

    Input :  sentence (str)   e.g. "I am here"
    Output:  list[str]        e.g. ['<.s>', '<.s>', 'i', 'am', 'here', '<./s>']
    """
    # TODO 1: lowercase the sentence and remove punctuation
    # TODO 2: split it into a list of words
    # TODO 3: return that list with TWO START tokens in front and ONE END token at the end
    pass

In [ ]:
def extract_ngrams(tokens, n):
    """
    Return all n-grams of a token list, as tuples.

    >>> extract_ngrams(['a', 'b', 'c', 'd'], 3)
    [('a', 'b', 'c'), ('b', 'c', 'd')]
    """
    # TODO: slide a window of width n across tokens and collect tuples
    pass

In [ ]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for sentence in corpus_sentences:
    # TODO 1: tokens = preprocess(sentence)
    # TODO 2: update each Counter with extract_ngrams(tokens, n) for n = 1, 2 and 3
    pass

# TODO 3: V = the number of DISTINCT tokens in the vocabulary
V = None

# TODO 4: total_tokens = the TOTAL number of tokens (needed in Task 5)
total_tokens = None

print(f"V = {V},  total tokens = {total_tokens}")
print(f"Count('i', 'want')       = {bigram_counts[('i', 'want')]}")
print(f"Count('i', 'want', 'to') = {trigram_counts[('i', 'want', 'to')]}")

---
# Task 2: Trigram Probability with Add-k Smoothing

### Concept

Plain counting (MLE) gives probability **0** to every trigram that never occurred. Since a
sentence probability is a product, one zero makes the whole sentence impossible, and
`log(0)` is `-inf`.

The tutorial fixed this by adding **1** to every count. But the `1` is not a magic number — it
is a value we choose, called a **hyperparameter**. The general form is **Add-k smoothing**:

$$P(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3) + k}{\text{Count}(w_1, w_2) + k \cdot V}$$

Setting `k = 1` recovers Add-1. Setting `k = 0` gives plain MLE with no smoothing.

### Task

Write `calculate_trigram_prob` implementing the Add-k formula.

- **Inputs:** a trigram tuple, the trigram and bigram count dictionaries, `V`, and `k`.
- **Output:** the smoothed probability.
- If `k = 0`, return the unsmoothed MLE instead.

**Hints**
- The history is `trigram[:2]` — a **bigram** tuple, so look it up in `bigram_counts`, **not**
  in `unigram_counts`.
- The denominator is `+ k*V`, **not** `+ k`.
- Use `.get(key, 0)` so unseen n-grams give 0 rather than a `KeyError`.
- For `k = 0`, guard against dividing by zero: if the history count is 0, return `0.0`.

In [ ]:
def calculate_trigram_prob(trigram, trigram_counts, bigram_counts, V, k=1.0):
    """
    Add-k smoothed P(w3 | w1, w2). k=1.0 gives Add-1, k=0 gives unsmoothed MLE.

    Input :  trigram         tuple of 3 tokens, e.g. ('i', 'want', 'to')
             trigram_counts  Counter of trigrams
             bigram_counts   Counter of bigrams
             V               vocabulary size (int)
             k               smoothing constant (float)
    Output:  float           the probability
    """
    # TODO 1: history = the first two tokens of the trigram (this is a BIGRAM key)
    # TODO 2: look up the trigram count and the history count, using .get(..., 0)
    # TODO 3: if k == 0, return the unsmoothed MLE (0.0 if the history count is 0)
    # TODO 4: otherwise return (trigram_count + k) / (history_count + k * V)
    pass


# a seen trigram and an unseen one, with and without smoothing
print(calculate_trigram_prob(('i', 'want', 'to'),    trigram_counts, bigram_counts, V, k=1))
print(calculate_trigram_prob(('i', 'want', 'pizza'), trigram_counts, bigram_counts, V, k=1))
print(calculate_trigram_prob(('i', 'want', 'pizza'), trigram_counts, bigram_counts, V, k=0))

---
# Task 3: Sentence Log Probability

Write `calculate_sentence_log_prob_trigram`, which computes the total **log probability** of a
sentence.

- Preprocess the sentence for a trigram model (as in Task 1).
- Break it into its trigrams.
- For each trigram, call `calculate_trigram_prob`.
- **Sum** the log probabilities.

**Example:** the log probability of `"I want food"` is the sum of the logs of

`P(I | <.s>, <.s>) · P(want | <.s>, I) · P(food | I, want) · P(<./s> | want, food)`

We add logs instead of multiplying probabilities to avoid **underflow** — the product of many
small numbers becomes too small for the computer to store.

**Hints**
- Reuse `preprocess` and `extract_ngrams(tokens, 3)`.
- Start the running total at `0.0`, not `1.0`.
- Use `math.log(p, 2)` — base 2, because Task 4 depends on it.
- If any probability is `0` (possible when `k = 0`), return `-float("inf")` immediately.

In [ ]:
def calculate_sentence_log_prob_trigram(sentence, trigram_counts, bigram_counts, V, k=1.0):
    """
    Input :  sentence (str)
    Output:  float   the total log2 probability of the sentence
    """
    # TODO 1: tokens = preprocess(sentence)
    # TODO 2: total = 0.0
    # TODO 3: for each trigram in extract_ngrams(tokens, 3):
    #             get its probability with calculate_trigram_prob
    #             if the probability is 0, return -float("inf")
    #             add math.log(prob, 2) to the total
    # TODO 4: return the total
    pass


for s in test_sentences:
    print(f"{s:<36} log prob = {calculate_sentence_log_prob_trigram(s, trigram_counts, bigram_counts, V)}")

---
# Task 4: Perplexity

### Concept

A good language model assigns high probability to sentences it has not seen. **Perplexity**
measures how "surprised" the model is by a test sentence.

- **High perplexity:** the model was very surprised. (Bad model.)
- **Low perplexity:** the model was not surprised. (Good model.)

A log probability cannot be compared between sentences of different lengths, because a longer
sentence always scores lower. Perplexity normalizes by the number of words `N`:

$$PP(W) = P(w_1 w_2 \dots w_N)^{-1/N} \;=\; 2^{-L/N}$$

- `L` is the log probability from Task 3.
- `N` is the number of tokens **including** the `<./s>` token but **excluding** the initial
  `<.s>` tokens. For `"I want food"`, `N = 4`.

### Task

Write `calculate_perplexity`, which takes a sentence, computes `L` and `N`, and returns
`2 ** (-L / N)`.

**Hints**
- `N` is the number of **words** plus 1. Do **not** use `len(preprocess(sentence))` — that
  would wrongly include the two `<.s>` tokens.
- If `L` is `-inf`, return `float("inf")`.

In [ ]:
def calculate_perplexity(sentence, trigram_counts, bigram_counts, V, k=1.0):
    """
    Input :  sentence (str)
    Output:  float   the perplexity, 2 ** (-L / N)
    """
    # TODO 1: L = calculate_sentence_log_prob_trigram(...)
    # TODO 2: N = number of words in the sentence + 1 (for the <./s> token)
    # TODO 3: if L is -inf, return float("inf")
    # TODO 4: return 2 ** (-L / N)
    pass


for s in test_sentences:
    print(f"{s:<36} perplexity = {calculate_perplexity(s, trigram_counts, bigram_counts, V)}")

---
# Task 5: Linear Interpolation

### Concept

Add-k has a weakness. It treats **every** unseen trigram exactly the same way. But consider
these two trigrams, neither of which is in the corpus:

- `('want', 'to', 'eat')` — even if this exact trigram were missing, `('to', 'eat')` is a very
  common pair, so the model should still consider it likely.
- `('credit', 'card', 'eat')` — nothing about this is plausible at any level.

Add-k gives them the same treatment. It has no way to use the fact that the shorter context
`('to', 'eat')` was seen often.

**Linear interpolation** fixes this by mixing all three models together. Instead of relying on
the trigram estimate alone, take a weighted average of the trigram, bigram and unigram
estimates:

$$P_{\text{interp}}(w_3 \mid w_1, w_2) = \lambda_1 P_{ML}(w_3 \mid w_1, w_2)
+ \lambda_2 P_{ML}(w_3 \mid w_2) + \lambda_3 P_{ML}(w_3)$$

where each piece is a plain **unsmoothed MLE** estimate:

$$P_{ML}(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3)}{\text{Count}(w_1, w_2)}
\qquad
P_{ML}(w_3 \mid w_2) = \frac{\text{Count}(w_2, w_3)}{\text{Count}(w_2)}
\qquad
P_{ML}(w_3) = \frac{\text{Count}(w_3)}{\text{total tokens}}$$

The weights must satisfy $\lambda_1 + \lambda_2 + \lambda_3 = 1$, which is what guarantees the
result is still a valid probability. The unigram term acts as a safety net: as long as
$\lambda_3 > 0$ and the word was seen at all, the result can never be zero.

### Task

Write four functions:

1. `ml_trigram_prob(trigram, ...)` — returns $P_{ML}(w_3 \mid w_1, w_2)$, or `0.0` if the
   history count is 0
2. `ml_bigram_prob(trigram, ...)` — returns $P_{ML}(w_3 \mid w_2)$, or `0.0` if the history
   count is 0
3. `ml_unigram_prob(trigram, ...)` — returns $P_{ML}(w_3)$
4. `calculate_interpolated_prob(trigram, ..., lambdas)` — combines the three using the weights

Then write `calculate_perplexity_interpolated`, which is the same as Task 4 but calls
`calculate_interpolated_prob` instead.

**Hints**
- For a trigram `(w1, w2, w3)`: the **bigram** you need is `trigram[1:]`, i.e. `(w2, w3)`, and
  its history is the unigram `(w2,)`. The **unigram** you need is `(w3,)`.
- Remember your counter keys are tuples, so a unigram key is `('the',)` and not `'the'`.
- `ml_unigram_prob` divides by `total_tokens`, not by `V`.
- Pass `lambdas` as a tuple `(l1, l2, l3)` and unpack it inside the function.
- These are **unsmoothed** MLE estimates. Each one can be 0 on its own — that is fine, because
  the weighted sum will still be non-zero as long as one component is non-zero.

In [ ]:
def ml_trigram_prob(trigram, trigram_counts, bigram_counts):
    """Unsmoothed P_ML(w3 | w1, w2) = Count(w1,w2,w3) / Count(w1,w2)."""
    # TODO: return 0.0 if the history count is 0, otherwise the ratio
    pass


def ml_bigram_prob(trigram, bigram_counts, unigram_counts):
    """Unsmoothed P_ML(w3 | w2) = Count(w2,w3) / Count(w2)."""
    # TODO 1: the bigram you need is trigram[1:], its history is the unigram (trigram[1],)
    # TODO 2: return 0.0 if the history count is 0, otherwise the ratio
    pass


def ml_unigram_prob(trigram, unigram_counts, total_tokens):
    """Unsmoothed P_ML(w3) = Count(w3) / total number of tokens."""
    # TODO: look up the unigram (trigram[2],) and divide by total_tokens
    pass

In [ ]:
def calculate_interpolated_prob(trigram, trigram_counts, bigram_counts,
                                unigram_counts, total_tokens, lambdas):
    """
    Linear interpolation of the trigram, bigram and unigram MLE estimates.

    Input :  lambdas   tuple (l1, l2, l3) which must sum to 1
    Output:  float     l1*P_ML(w3|w1,w2) + l2*P_ML(w3|w2) + l3*P_ML(w3)
    """
    # TODO 1: unpack lambdas into l1, l2, l3
    # TODO 2: compute the three MLE estimates using the functions above
    # TODO 3: return the weighted sum
    pass


def calculate_perplexity_interpolated(sentence, trigram_counts, bigram_counts,
                                      unigram_counts, total_tokens, lambdas):
    """Same as Task 4, but using the interpolated probability."""
    # TODO 1: tokens = preprocess(sentence), then loop over its trigrams
    # TODO 2: sum the log2 of calculate_interpolated_prob for each trigram
    #         if any probability is 0, return float("inf")
    # TODO 3: N = number of words + 1, then return 2 ** (-L / N)
    pass


# sanity check: the three lambdas must sum to 1
print(calculate_interpolated_prob(('i', 'want', 'to'), trigram_counts, bigram_counts,
                                  unigram_counts, total_tokens, (0.5, 0.3, 0.2)))

---
# Task 6: Tune Both Models and Compare

### 6a. Tune `k` for the Add-k model

Loop over `k = [0.01, 0.1, 0.5, 1, 2, 5, 10]`. For each `k`, compute the perplexity of every
test sentence and their average. Report the `k` with the lowest average perplexity.

### 6b. Tune the weights for the interpolation model

Search over all combinations of $(\lambda_1, \lambda_2, \lambda_3)$ in steps of `0.1` that sum
to 1. There are 66 such combinations. For each, compute the average perplexity over the test
sentences, and report the best combination.

### 6c. Compare

Print a short comparison of the two tuned models and state which one wins.

Then answer these two questions in a markdown cell:

1. What weight did the trigram component $\lambda_1$ receive in your best combination? Given
   that this is a *trigram* assignment, is that result surprising? What does it tell you about
   the amount of training data?
2. Some $(\lambda_1, \lambda_2, \lambda_3)$ combinations give an **infinite** perplexity.
   Which ones, and why?

**Hints**
- To build the grid, loop `i1` from 0 to 10 and `i2` from 0 to `10 - i1`, then set
  `i3 = 10 - i1 - i2`. Divide each by 10 to get the lambdas. This guarantees they sum to 1
  and avoids floating point problems.
- `min(results, key=results.get)` returns the key with the smallest value.
- Skip or ignore any combination that produces `float("inf")`.
- Format with `f"{value:10.2f}"` so the columns line up.

In [ ]:
# ---- 6a. Tune k for the Add-k model ----
k_values = [0.01, 0.1, 0.5, 1, 2, 5, 10]

# TODO 1: for each k, compute the perplexity of each test sentence and their average
# TODO 2: print a table and report the best k

In [ ]:
# ---- 6b. Tune the lambdas for the interpolation model ----

# TODO 1: build the list of all (l1, l2, l3) in steps of 0.1 that sum to 1  (66 of them)
# TODO 2: for each combination, compute the average perplexity over the test sentences
# TODO 3: report the best combination and its perplexity
# TODO 4: print the top few combinations so you can see the trend

In [ ]:
# ---- 6c. Compare the two tuned models ----

# TODO: print the best Add-k result and the best interpolation result side by side,
#       and state which model wins and by how much

### Your answers to 6c

**1. Weight of the trigram component:**

**2. Which lambda combinations give infinite perplexity, and why:**